# Face Anonymization with Auto-Detected Nasion (MediaPipe)

Drop-in alternative to `51_manual_5pt_anonymization.ipynb`. MediaPipe replaces
the click step: `detect_nasion_auto` runs the face landmarker on a sweep of
rendered views and back-projects the nasion onto the mesh; the remaining four
landmarks (`Iz, Cz, LPA, RPA`) are derived geometrically from the head shape.
The result rejoins the same `anonymize_scan` core, so the saved
`.obj + .mtl + .jpg + _landmarks.tsv` bundle is identical in shape to the
manual route.

Flow: load &rarr; auto-detect Nz (or fall back to manual picker) &rarr;
derive 4 landmarks &rarr; `anonymize_scan` &rarr; save.

MediaPipe is an optional extra. Install once with
`pip install -e .[anonymization]` from inside the cedalion repo.


In [ ]:
import logging
import os

import numpy as np
import pyvista as pv
import xarray as xr
from PIL import Image
import trimesh

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.vis.blocks
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.geometry.photogrammetry.anonymization import (
    anonymize_scan,
    save_anonymized_scan,
    detect_nasion_auto,
    derive_landmarks_from_nasion,
)

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === Options ===
SUBJECT_NUMBER = 17
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'
SCAN_PATH = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
OUT_PATH = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}_anon_auto.obj'

# True: drop into the manual picker if MediaPipe finds no face. False: raise.
MANUAL_FALLBACK = True

# anonymize_scan tunables (mm). Defaults match the library's recommended values.
ANON_OPTIONS = dict(
    head_isolation_radius_mm=220.0,
    cap_band_width_mm=15.0,
    cap_bin_size_mm=1.0,
    cap_foot_grad_threshold=0.2,
    cap_z_ceiling_mm=40.0,
    eyebrow_offset_mm=10.0,
    ear_delete_radius_mm=40.0,
    landmark_keep_radius_mm=8.0,
)


## Load the Einstar scan


In [ ]:
surface = cedalion.io.read_einstar_obj(SCAN_PATH)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

# trimesh 4.6 places the texture image on visual.material.image; if neither
# path has it, attach it from the sibling JPG so save_anonymized_scan can
# sanitize the final texture.
visual = surface.mesh.visual
img = getattr(visual, 'image', None) or getattr(getattr(visual, 'material', None), 'image', None)
if img is None:
    jpg = SCAN_PATH.replace('.obj', '.jpg')
    uv = getattr(visual, 'uv', None)
    assert os.path.exists(jpg) and uv is not None, 'no texture available'
    surface.mesh.visual = trimesh.visual.TextureVisuals(
        uv=uv, image=Image.open(jpg).convert('RGBA'),
    )


## Auto-detect the nasion

`detect_nasion_auto` renders the mesh from a yaw/pitch sweep, runs the
MediaPipe Face Landmarker on each view, and back-projects the nasion of the
best frontal hit onto the surface. Returns `None` if no view validates -- the
fallback below opens the manual picker just for the nasion.


In [ ]:
result = detect_nasion_auto(surface)

if result is None:
    if not MANUAL_FALLBACK:
        raise RuntimeError(
            'MediaPipe could not find a face; rerun notebook 51 (manual 5pt) '
            'or set MANUAL_FALLBACK = True.'
        )
    print('MediaPipe failed -- falling back to manual nasion click.')
    pvplt = pv.Plotter()
    get_landmarks = cedalion.vis.blocks.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True,
    )
    pvplt.show()
    picked = get_landmarks()
    nz_label_idx = list(picked['label'].values).index('Nz')
    nasion = np.asarray(
        picked.pint.dequantify().isel(label=nz_label_idx).values,
        dtype=float,
    )
    detector_meta = {'method': 'manual_fallback', 'confidence': None}
else:
    nasion, detector_meta = result
    nasion = np.asarray(nasion, dtype=float)

print(f'Nz = {nasion} (method={detector_meta.get("method")}, '
      f'confidence={detector_meta.get("confidence")})')


## Derive `Iz`, `Cz`, `LPA`, `RPA` geometrically

Given a known nasion, the remaining four landmarks come from extrema of the
head in the axis-normalized frame: `Cz` is the apex, `Iz` the rear midline,
and `LPA`/`RPA` are snapped onto the side of the head at nasion height.
Output is a `LabeledPoints` in the input surface's frame -- exactly what the
manual picker would have returned.


In [ ]:
landmarks = derive_landmarks_from_nasion(surface, nasion)
labels = list(landmarks['label'].values)
assert set(labels) == {'Nz', 'Iz', 'Cz', 'LPA', 'RPA'}, f'bad labels: {labels}'
display(landmarks)


## Anonymize


In [ ]:
surface_anon, landmarks_anon = anonymize_scan(surface, landmarks, **ANON_OPTIONS)
print(f'S{SUBJECT_NUMBER}: {surface.nvertices:,} -> {surface_anon.nvertices:,} verts '
      f'(-{surface.nvertices - surface_anon.nvertices:,})')


## Compare original vs auto-anonymized

Both meshes live in the `digitized` frame, so the side-by-side view shares
one coordinate system. The same five landmark spheres are overlaid on each.


In [ ]:
lm_colors = {
    'Nz': 'lime', 'Iz': 'magenta', 'Cz': 'cyan',
    'Lpa': 'orange', 'Rpa': 'blue', 'LPA': 'orange', 'RPA': 'blue',
}
n_removed = surface.nvertices - surface_anon.nvertices

orig_vtk = trimesh_to_vtk_polydata(surface.mesh)
anon_vtk = trimesh_to_vtk_polydata(surface_anon.mesh)

lm_pos = landmarks_anon.pint.dequantify().values
lm_lbls = landmarks_anon['label'].values

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
pvplt.add_mesh(pv.wrap(orig_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(lm_lbls, lm_pos):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(lm_lbls, lm_pos):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(
    f'S{SUBJECT_NUMBER} Auto-Nz Anonymized (-{n_removed:,} verts)',
    position='upper_left', font_size=14,
)

pvplt.link_views()
pvplt.show()


## Save


In [ ]:
written = save_anonymized_scan(surface_anon, OUT_PATH, landmarks=landmarks_anon)
for p in written:
    print(f'wrote {p}')
